AAPL — 1-minute bars → bronze
---
---


**Table of contents**<a id='toc0_'></a>    
- 1. [Introduction](#toc1_)    
- 2. [Utils](#toc2_)    
  - 2.1. [Libraries & paths](#toc2_1_)    
  - 2.2. [Parameters](#toc2_2_)    
- 3. [Support tables](#toc3_)    
  - 3.1. [`dim_market_calendar`](#toc3_1_)    
  - 3.2. [`dim_corporate_actions`](#toc3_2_)    
- 4. [Bronze — `fact_bars_raw`](#toc4_)    
- 5. [Verification](#toc5_)    
- 6. [Catalog](#toc6_)    

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->
---

# 1. <a id='toc1_'></a>[Introduction](#toc0_)
Extraction of every field Alpaca exposes for AAPL 1-minute bars, from 2016 to today,
into `lakehouse.bronze` of the Unity Catalog lakehouse.

Fixed decisions (§1 of the design document):

| Decision | Value | Why |
|---|---|---|
| Source | Alpaca Market Data v2, `feed=sip` | 100% of the volume; available on the Basic plan for data older than 15 minutes. |
| Granularity | `1Min` | ~1M rows for AAPL 2016→2026, re-aggregable to 5m/15m/1h/1d without touching the API again. |
| Adjustment | `raw` | Alpaca adjusts with the factors known *today*; storing raw keeps the dataset reproducible. Adjustment happens in silver. |
| Format | Delta Lake, external table in Unity Catalog | Atomic per-partition commits make a retry safe; the catalog makes the same table readable from pandas, Spark and the UC UI. |
| Timezone | UTC | `America/New_York` only appears in silver, where the market calendar makes local time meaningful. |

The layer boundary matters: a bug in a feature definition must never force a
re-download of ten years of history through a 200 req/min budget.

---

# 2. <a id='toc2_'></a>[Utils](#toc0_)

## 2.1. <a id='toc2_1_'></a>[Libraries & paths](#toc0_)

In [ ]:
import logging
import os
import pathlib
import sys

from dotenv import load_dotenv
MAIN = os.path.join(os.path.abspath(os.path.curdir), 'work/')
DATA_ETL_ROOT = os.path.join(MAIN, 'data-etl')
print(DATA_ETL_ROOT)

sys.path.insert(0, str(DATA_ETL_ROOT))

load_dotenv(MAIN)
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s")

from src.dwh.bronze import alpaca_bars, alpaca_reference
from src.extractions.alpaca import MarketData
from src.storage.catalog import get_catalog

## 2.2. <a id='toc2_2_'></a>[Parameters](#toc0_)

In [ ]:
from datetime import date, timedelta

SYMBOLS = "AAPL"          # the schema is multi-symbol from day one: "AAPL,SPY,QQQ" also works
TIMEFRAME = "1Min"
FEED = "sip"
ADJUSTMENT = "raw"
START_DATE = date(2016, 1, 1)
END_DATE = date.today() - timedelta(days=1) # extract until yesterday, last 15 minutes are not allowed in free plan

CREDENTIALS = {
    "APCA-API-KEY-ID": os.getenv("APCA-API-KEY-ID"),
    "APCA-API-SECRET-KEY": os.getenv("APCA-API-SECRET-KEY"),
}

# UNITY_CATALOG_URI / UNITY_CATALOG_NAME / LAKEHOUSE_ROOT come from the environment.
lakehouse = get_catalog()
market = MarketData(credentials=CREDENTIALS)

print(f"{SYMBOLS} {TIMEFRAME} {START_DATE} \u2192 {END_DATE} (feed={FEED}, adjustment={ADJUSTMENT})")
print(f"catalog {lakehouse.catalog} at {lakehouse.client.uri}, tables under {lakehouse.warehouse_root}")

---

# 3. <a id='toc3_'></a>[Support tables](#toc0_)

## 3.1. <a id='toc3_1_'></a>[`dim_market_calendar`](#toc0_)

Without it there is no way to tell "a minute with no trades" from "the market was
closed", and that distinction decides how gaps are filled in silver. Early closes at
13:00 ET (Thanksgiving eve, 24 Dec) are real and frequent.

In [ ]:
calendar = alpaca_reference.ingest_market_calendar(
    market, start=START_DATE, end=END_DATE, lakehouse=lakehouse
)

print(f"{len(calendar)} sessions, {int(calendar['is_half_day'].sum())} of them early closes")
calendar.head()

## 3.2. <a id='toc3_2_'></a>[`dim_corporate_actions`](#toc0_)

AAPL in this range: the 4:1 split of 2020-08-31 and quarterly dividends. Both create
discontinuities that would otherwise be learned as signal.

In [ ]:
corporate_actions = alpaca_reference.ingest_corporate_actions(
    market, symbols=SYMBOLS, start=START_DATE, end=END_DATE, lakehouse=lakehouse
)

corporate_actions.groupby("type").size()

In [ ]:
corporate_actions[corporate_actions["type"] == "split"]

---

# 4. <a id='toc4_'></a>[Bronze — `fact_bars_raw`](#toc0_)

One request per month (`limit=10000`, `next_page_token`), each write aligned with exactly one
`symbol=/year=/month=` partition. Partitions are rewritten whole rather than appended to,
so a retry after a partial failure cannot duplicate rows.

Set `is_overwrite=False` to resume an interrupted backfill without re-hitting the API.

In [ ]:
summary = alpaca_bars.ingest_bars_raw(
    market,
    symbols=SYMBOLS,
    start=START_DATE,
    end=END_DATE,
    timeframe=TIMEFRAME,
    feed=FEED,
    adjustment=ADJUSTMENT,
    lakehouse=lakehouse,
    is_overwrite=True,
)

print(f"{summary['row_count'].sum():,} bars written across {len(summary)} monthly partitions")
summary.tail()

---

# 5. <a id='toc5_'></a>[Verification](#toc0_)

In [ ]:
bars = alpaca_bars.read_bars_raw(symbol="AAPL", lakehouse=lakehouse)

print(f"{len(bars):,} rows, {bars['timestamp_at'].min()} \u2192 {bars['timestamp_at'].max()}")
bars.groupby("year").size()

In [ ]:
# Every field the API returns, plus the request provenance that makes an audit possible.
bars.head()

In [ ]:
# The logical key (symbol, timestamp_at, feed) must be unique for the ingestion to be idempotent.
duplicate_count = bars.duplicated(subset=["symbol", "timestamp_at", "feed"]).sum()
print(f"duplicates on (symbol, timestamp_at, feed): {duplicate_count}")

---

# 6. <a id='toc6_'></a>[Catalog](#toc0_)

The three tables are external Delta tables registered in Unity Catalog, so Spark
reads them as `lakehouse.bronze.<table>` without knowing any path.


In [ ]:
for table in lakehouse.client.list_tables(lakehouse.catalog, "bronze"):
    print(f"{lakehouse.catalog}.bronze.{table['name']:24} {table['storage_location']}")